In [34]:
import os
import re
import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

from imblearn.over_sampling import RandomOverSampler

from collections import Counter

import joblib

In [35]:
DATASET_PATH = r"../../Dataset/ADFA_dataset/ADFA-WD-SAA_Master/ADFA-WD-SAA_Master/Full_Process_Traces"

In [36]:
print(os.path.exists(DATASET_PATH))

True


In [37]:
all_data = []

all_labels = []

In [38]:
# =========================================
# READ WINDOWS ADVANCED DATASET
# =========================================

print("Reading Windows Advanced Dataset...\n")

all_data = []

all_labels = []

for root, dirs, files in os.walk(DATASET_PATH):

    for file in tqdm(files):

        # =========================================
        # READ ONLY GHC FILES
        # =========================================

        if file.endswith(".GHC"):

            file_path = os.path.join(root, file)

            try:

                with open(
                    file_path,
                    "r",
                    encoding="utf-8",
                    errors="ignore"
                ) as f:

                    # =========================================
                    # READ DLL/API TOKENS
                    # =========================================

                    content = f.read().strip()

                    tokens = content.split()

                    # =========================================
                    # SKIP EMPTY FILES
                    # =========================================

                    if len(tokens) > 0:

                        all_data.append(tokens)

                        # =========================================
                        # EXTRACT FOLDER NAME
                        # =========================================

                        folder_name = os.path.basename(root)

                        # =========================================
                        # GROUP SESSION LABELS
                        # =========================================
                        # S1-1 -> S1
                        # S2-5 -> S2
                        # S4-8 -> S4
                        # =========================================

                        label = folder_name.split("-")[0]

                        all_labels.append(label)

            except Exception as e:

                print("\nError Reading File:")

                print(file_path)

                print("Exception:", e)

print("\n===================================")
print("Dataset Loading Complete")
print("===================================")

print(f"Total Samples: {len(all_data)}")

print(f"Total Labels: {len(all_labels)}")

Reading Windows Advanced Dataset...



0it [00:00, ?it/s]
0it [00:00, ?it/s]
100%|██████████| 35/35 [00:00<00:00, 670.13it/s]
0it [00:00, ?it/s]
100%|██████████| 23/23 [00:00<00:00, 260.33it/s]
0it [00:00, ?it/s]
100%|██████████| 15/15 [00:00<00:00, 421.22it/s]
0it [00:00, ?it/s]
100%|██████████| 11/11 [00:00<00:00, 353.75it/s]


Dataset Loading Complete
Total Samples: 862
Total Labels: 862


In [39]:
print("\nUnique Labels:\n")

print(set(all_labels))


Unique Labels:

{'S3', 'S1', 'S4', 'S2'}


In [40]:
all_tokens = []

for seq in all_data:

    all_tokens.extend(seq)

unique_tokens = list(set(all_tokens))

print("Unique API Calls:", len(unique_tokens))

Unique API Calls: 621


In [41]:
token_encoder = LabelEncoder()

token_encoder.fit(unique_tokens)

LabelEncoder()

In [42]:
encoded_sequences = []

for seq in all_data:

    encoded_seq = token_encoder.transform(seq)

    encoded_sequences.append(encoded_seq)

In [43]:
MAX_LEN = 1000

In [44]:
processed_data = []

for seq in encoded_sequences:

    seq = list(seq)

    if len(seq) >= MAX_LEN:

        seq = seq[:MAX_LEN]

    else:

        seq = seq + [0] * (MAX_LEN - len(seq))

    processed_data.append(seq)

In [45]:
X = np.array(processed_data)

y = np.array(all_labels)

print(X.shape)

print(y.shape)

(862, 1000)
(862,)


In [46]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

In [47]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y_encoded,

    test_size=0.2,

    random_state=42,

    stratify=y_encoded
)

In [48]:
print("\nBefore Balancing:\n")

print(Counter(y_train))


Before Balancing:

Counter({np.int64(0): 293, np.int64(1): 169, np.int64(3): 114, np.int64(2): 113})


In [49]:
ros = RandomOverSampler(
    random_state=42
)

X_train, y_train = ros.fit_resample(
    X_train,
    y_train
)

In [50]:
print("\nAfter Balancing:\n")

print(Counter(y_train))


After Balancing:

Counter({np.int64(1): 293, np.int64(0): 293, np.int64(2): 293, np.int64(3): 293})


In [51]:
model = XGBClassifier(

    n_estimators=150,

    max_depth=4,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    objective='multi:softprob',

    num_class=len(np.unique(y_encoded)),

    eval_metric='mlogloss',

    n_jobs=-1,

    random_state=42
)

In [52]:
model.fit(

    X_train,
    y_train,

    eval_set=[(X_test, y_test)],

    verbose=True
)

[0]	validation_0-mlogloss:1.35951
[1]	validation_0-mlogloss:1.33469
[2]	validation_0-mlogloss:1.31024
[3]	validation_0-mlogloss:1.28713
[4]	validation_0-mlogloss:1.26631
[5]	validation_0-mlogloss:1.24955
[6]	validation_0-mlogloss:1.23242
[7]	validation_0-mlogloss:1.21617
[8]	validation_0-mlogloss:1.20348
[9]	validation_0-mlogloss:1.19176
[10]	validation_0-mlogloss:1.17856
[11]	validation_0-mlogloss:1.16673
[12]	validation_0-mlogloss:1.15761
[13]	validation_0-mlogloss:1.14807
[14]	validation_0-mlogloss:1.13864
[15]	validation_0-mlogloss:1.12899
[16]	validation_0-mlogloss:1.12110
[17]	validation_0-mlogloss:1.11446
[18]	validation_0-mlogloss:1.10795
[19]	validation_0-mlogloss:1.10150
[20]	validation_0-mlogloss:1.09402
[21]	validation_0-mlogloss:1.08965
[22]	validation_0-mlogloss:1.08324
[23]	validation_0-mlogloss:1.07753
[24]	validation_0-mlogloss:1.07235
[25]	validation_0-mlogloss:1.06751
[26]	validation_0-mlogloss:1.06332
[27]	validation_0-mlogloss:1.05838
[28]	validation_0-mlogloss:1.0

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [53]:
y_prob = model.predict_proba(X_test)

y_pred = np.argmax(y_prob, axis=1)

In [54]:
train_pred = model.predict(X_train)

train_accuracy = accuracy_score(
    y_train,
    train_pred
)

print(f"\nTraining Accuracy: {train_accuracy:.4f}")


Training Accuracy: 0.7910


In [55]:
accuracy = accuracy_score(y_test, y_pred)

print(f"\nTest Accuracy: {accuracy:.4f}")


Test Accuracy: 0.5202


In [56]:
print(

    classification_report(

        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.83      0.66      0.73        73
           1       0.70      0.37      0.48        43
           2       0.29      0.43      0.35        28
           3       0.27      0.48      0.35        29

    accuracy                           0.52       173
   macro avg       0.52      0.49      0.48       173
weighted avg       0.62      0.52      0.54       173



In [57]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

[[48  3 10 12]
 [ 7 16  7 13]
 [ 1  3 12 12]
 [ 2  1 12 14]]


In [58]:
os.makedirs(

    "../../trained_models/windows_advanced",

    exist_ok=True
)

joblib.dump(

    model,

    "../../trained_models/windows_advanced/windows_advanced_xgboost.pkl"
)

['../../trained_models/windows_advanced/windows_advanced_xgboost.pkl']

In [59]:
joblib.dump(

    token_encoder,

    "../../trained_models/windows_advanced/token_encoder.pkl"
)

joblib.dump(

    label_encoder,

    "../../trained_models/windows_advanced/label_encoder.pkl"
)

['../../trained_models/windows_advanced/label_encoder.pkl']

In [60]:
print("===================================")

print("AEGIS Windows Advanced IDS Complete")

print("===================================")

AEGIS Windows Advanced IDS Complete
